# BattSimX Studio: train and validate on the Panasonic 18650PF dataset
Dataset: P. Kollmeyer, *Panasonic 18650PF Li-ion Battery Data*, Mendeley Data v1 (2018), DOI 10.17632/wykht8y7tg.1
https://data.mendeley.com/datasets/wykht8y7tg  (cite it if you use it)

Run the cells top to bottom. **Stages:** 1 get data, 2 inspect, 3 preprocess, 4 split, 5 train physics model, 6 train ML model, 7 test, 8 save.
Predicts **cell temperature** during drive cycles. Voltage is used as an input, not a prediction target.

## 1. Get the data
One cell downloads the dataset straight into Colab (about 10 seconds) and unzips it. No upload needed.

In [ ]:
import os, zipfile, subprocess
DATA_DIR = "/content/data"
ZIP_PATH = "/content/panasonic.zip"
os.makedirs(DATA_DIR, exist_ok=True)
if not (os.path.exists(ZIP_PATH) and os.path.getsize(ZIP_PATH) > 150e6):
    subprocess.run(["wget", "-q", "-O", ZIP_PATH, "https://data.mendeley.com/public-api/zip/wykht8y7tg/download/1"], check=True)
print("zip size (MB):", round(os.path.getsize(ZIP_PATH)/1e6, 1), "(expected about 190)")
zipfile.ZipFile(ZIP_PATH).extractall(DATA_DIR)
for root, _, fs in os.walk(DATA_DIR):          # unpack any ZIPs inside the ZIP
    for f in fs:
        if f.lower().endswith(".zip"): zipfile.ZipFile(os.path.join(root, f)).extractall(root)
n_mat = sum(f.lower().endswith(".mat") for _, _, fs in os.walk(DATA_DIR) for f in fs)
print("data folder:", DATA_DIR, "|", n_mat, ".mat files")

In [ ]:
# FALLBACK, only if the cell above fails: upload the ZIP yourself, then run this cell
# from google.colab import files; up = files.upload()
# import zipfile; [zipfile.ZipFile(n).extractall(DATA_DIR) for n in up if n.lower().endswith(".zip")]

## 2. Inspect the files
Look at what is inside before trusting the loader. Read the dataset's readme too.

In [ ]:
import glob, os
files_all = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.mat"), recursive=True) +
                   glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
print(len(files_all), "files found")
for f in files_all[:40]: print(" ", os.path.relpath(f, DATA_DIR), f"{os.path.getsize(f)/1e6:.1f} MB")

In [ ]:
import numpy as np, pandas as pd
def peek(path):
    if path.lower().endswith(".csv"):
        d = pd.read_csv(path, nrows=5); print(path, "\n", d.head()); return
    from scipy.io import loadmat
    m = loadmat(path, squeeze_me=True, struct_as_record=False)
    keys = [k for k in m if not k.startswith("__")]
    print(os.path.basename(path), "variables:", keys)
    for k in keys:
        v = m[k]
        if hasattr(v, "_fieldnames"):
            print("  struct", k, "fields:", {f: np.asarray(getattr(v, f)).shape for f in v._fieldnames})
peek(files_all[0])

## 3. Preprocess
- read time, voltage, current, cell temperature, chamber temperature
- resample onto a **5 second grid** (keeps the simulation fast, and gives equal time steps)
- drop NaNs, compute amp-hours and the running heat integral
- ambient (chamber) temperature: taken from the **folder name** (e.g. `-10degC`), because the chamber column inside the files did not match the folders in the first run; the column value is kept as `Tamb_file` for comparison

**Edit `FILE_FILTER`** to keep only the drive-cycle files (not HPPC or impedance tests), using words that appear in their file names.

In [ ]:
import re
FILE_FILTER = r"drive cycles"   # regex on the file path; keeps only the drive-cycle tests ("" keeps every file)
DT = 5.0                  # seconds per step after resampling
MIN_ROWS = 600            # skip files shorter than 10 minutes
ALIASES = dict(
    time=["time", "t", "time_s", "test_time", "timestamp"],
    voltage=["voltage", "v", "volt", "voltage_v"],
    current=["current", "i", "amps", "current_a"],
    temp=["battery_temp_degc", "temperature", "temp", "cell_temp", "battery_temp", "temp_degc"],
    chamber=["chamber_temp_degc", "ambient", "chamber_temp"],
)
CELL = dict(mass=0.0475, cp=900.0)   # kg (Panasonic NCR18650PF about 47.5 g), J/kgK assumed. CHECK mass on datasheet.

def pick(df, key):
    cols = {c.lower(): c for c in df.columns}
    for a in ALIASES[key]:
        if a in cols: return cols[a]
    return None

def read_raw(path):
    if path.lower().endswith(".csv"):
        return pd.read_csv(path)
    from scipy.io import loadmat
    m = loadmat(path, squeeze_me=True, struct_as_record=False)
    s = m.get("meas")
    if s is None:                      # fall back: first struct in the file
        for k, v in m.items():
            if hasattr(v, "_fieldnames"): s = v; break
    if s is None: raise ValueError("no struct found; see the inspect cell")
    cols = {f: np.asarray(getattr(s, f)).ravel() for f in s._fieldnames}
    n = max(len(v) for v in cols.values())
    return pd.DataFrame({k: v for k, v in cols.items() if len(v) == n})

def load_clean(path):
    df = read_raw(path)
    mp = {k: pick(df, k) for k in ALIASES}
    need = [k for k in ("time", "voltage", "current", "temp") if mp[k] is None]
    if need: raise ValueError(f"missing columns {need}; found {list(df.columns)}")
    d = pd.DataFrame({"t": df[mp["time"]], "V": df[mp["voltage"]], "I": df[mp["current"]], "T": df[mp["temp"]]}).astype(float)
    d = d.replace([np.inf, -np.inf], np.nan).dropna().sort_values("t").drop_duplicates("t")
    rel = os.path.relpath(path, DATA_DIR)
    g = re.search(r"(-?\d+)\s*degc", rel.lower())
    Tfile = float(df[mp["chamber"]].astype(float).median()) if mp["chamber"] else np.nan
    Tamb = float(g.group(1)) if g else (Tfile if not np.isnan(Tfile) else float(d["T"].iloc[:50].mean()))
    if len(d) < MIN_ROWS: raise ValueError(f"too short ({len(d)} rows)")
    gap = float(np.max(np.diff(d["t"].values)))
    if gap > 300: raise ValueError(f"time gap of {gap/60:.0f} min inside the file (logger paused or time column jumps)")
    t0 = d["t"].iloc[0]; grid = np.arange(0, d["t"].iloc[-1] - t0, DT)
    out = pd.DataFrame({"t": grid})
    for c in ("V", "I", "T"): out[c] = np.interp(grid, d["t"].values - t0, d[c].values)
    out["Tamb"] = Tamb; out["Tamb_file"] = Tfile
    out["file"] = rel
    return out

import time
todo = [f for f in files_all if (not FILE_FILTER or re.search(FILE_FILTER, f, re.I))]
print(len(todo), "files match the filter; loading...")
data, skipped = [], []
t_start = time.time()
for i, f in enumerate(todo, 1):
    try: data.append(load_clean(f))
    except Exception as e: skipped.append((os.path.basename(f), str(e)[:90]))
    if i % 5 == 0 or i == len(todo): print(f"  {i}/{len(todo)} files read ({time.time()-t_start:.0f} s)", flush=True)
print(len(data), "files loaded;", len(skipped), "skipped")
for s in skipped[:15]: print("  skipped", s)
summary = pd.DataFrame([{"file": g["file"].iloc[0], "Tamb_C": g["Tamb"].iloc[0], "Tamb_file_C": g["Tamb_file"].iloc[0], "minutes": len(g)*DT/60,
                         "T_start": g["T"].iloc[0], "T_max": g["T"].max(), "T_rise": g["T"].max()-g["T"].iloc[0],
                         "I_min": g["I"].min(), "I_max": g["I"].max()} for g in data])
print("files per ambient temperature:"); print(summary.groupby("Tamb_C").size().to_string())
print("expected: about 10 files at each of -20, -10, 0, 10, 25. Anything else means the filter or loader needs a look.")
summary

## 4. Split into train and test
`SPLIT = "by_temperature"` holds out **every file at one chamber temperature** (`TEST_TEMP`), so the model is tested at a temperature it never saw. This is the stricter test, and the one to show judges.
`SPLIT = "random_files"` holds out about a quarter of the files at random.

In [ ]:
SPLIT = "by_temperature"      # or "random_files"
TEST_TEMP = None              # None = hold out the coldest temperature present; or set e.g. 10

assert len(data) >= 4, "need at least 4 usable files; check FILE_FILTER and the skipped list above"
temps = sorted({g["Tamb"].iloc[0] for g in data})
print("chamber temperatures present:", temps)
if SPLIT == "by_temperature":
    held = TEST_TEMP if TEST_TEMP is not None else temps[0]
    test = [g for g in data if abs(g["Tamb"].iloc[0] - held) < 1.5]
    train = [g for g in data if abs(g["Tamb"].iloc[0] - held) >= 1.5]
    print("held-out temperature:", held)
else:
    rng = np.random.default_rng(0); order = rng.permutation(len(data)); nt = max(2, len(data)//4)
    test = [data[i] for i in order[:nt]]; train = [data[i] for i in order[nt:]]
assert train and test, "empty train or test set; change SPLIT or TEST_TEMP"
split_df = pd.DataFrame({"file": [g["file"].iloc[0] for g in train+test], "Tamb_C": [g["Tamb"].iloc[0] for g in train+test],
                         "set": ["train"]*len(train)+["test"]*len(test)})
print(len(train), "train files,", len(test), "test files")
split_df

## 5. Model 1: physics model, calibrated
Lumped cell: `C dT/dt = I²·R(T) − hA·(T − T_ambient)`, with resistance that rises as the cell gets colder: `R(T) = R25 · exp(−k·(T − 25))`.
Three values are fitted to the training files by least squares: **R25** (ohm at 25 C), **k** (1/C) and **hA** (W/K). Mass and heat capacity come from the datasheet (CELL above).

In [ ]:
from scipy.optimize import least_squares
C_th = CELL["mass"] * CELL["cp"]

import math, time
try:
    from numba import njit
    @njit(cache=False)
    def _sim(I, Ta, T0, R25, kT, hA, C, dt):
        n = len(I); T = np.empty(n); T[0] = T0
        for k in range(1, n):
            R = R25 * math.exp(-kT * (T[k-1] - 25.0))
            T[k] = T[k-1] + dt * (I[k-1] * I[k-1] * R - hA * (T[k-1] - Ta[k-1])) / C
        return T
    print("using numba (fast)")
except ImportError:
    def _sim(I, Ta, T0, R25, kT, hA, C, dt):          # plain-Python fallback on lists
        I = I.tolist(); Ta = Ta.tolist(); T = [T0] * len(I)
        for k in range(1, len(I)):
            R = R25 * math.exp(-kT * (T[k-1] - 25.0))
            T[k] = T[k-1] + dt * (I[k-1] * I[k-1] * R - hA * (T[k-1] - Ta[k-1])) / C
        return np.array(T)
    print("numba not found; using the slower fallback")

def simulate(g, R25, kT, hA):
    return _sim(g["I"].values.astype(float), g["Tamb"].values.astype(float), float(g["T"].iloc[0]),
                float(R25), float(kT), float(hA), float(C_th), float(DT))

def resid(p): return np.concatenate([simulate(g, *p) - g["T"].values for g in train])
t_fit = time.time()
fit = least_squares(resid, x0=[0.04, 0.02, 0.05], bounds=([1e-3, 0.0, 1e-3], [0.5, 0.1, 1.0]), max_nfev=40, verbose=2)
print(f"fit took {time.time()-t_fit:.0f} s")
R_fit, k_fit, hA_fit = fit.x; PHYS = (R_fit, k_fit, hA_fit)
print(f"fitted R25 = {R_fit*1000:.1f} mOhm, k = {k_fit:.3f} /C, hA = {hA_fit:.4f} W/K   (train RMSE {np.sqrt(np.mean(fit.fun**2)):.2f} C)")

## 6. Model 2: machine-learning surrogate
Gradient-boosted trees learn the **error of the physics model** (measured minus simulated temperature) from current, current², voltage, ambient temperature and the physics temperature. Final ML prediction = physics prediction + learned correction.
(The first run used time and cumulative amp-hours as inputs; the model then failed outside its training range, so those are removed.)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
def feats(g, p_phys):
    return pd.DataFrame({"I": g["I"].values, "I2": g["I"].values**2, "V": g["V"].values,
                         "Tamb": g["Tamb"].values, "Tphys_rel": p_phys - g["Tamb"].values})
phys_train = [simulate(g, *PHYS) for g in train]
Xtr = pd.concat([feats(g, p) for g, p in zip(train, phys_train)])
ytr = np.concatenate([g["T"].values - p for g, p in zip(train, phys_train)])
ml = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_depth=4, random_state=0).fit(Xtr, ytr)
print("trained on", len(Xtr), "rows from", len(train), "files")

## 7. Test on the held-out files
Compared with a **baseline** that predicts the cell never warms up. A model is useful only if it clearly beats the baseline.

In [ ]:
def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2)))
rows, P, M, B, Y = [], [], [], [], []
for g in test:
    y = g["T"].values; p_phys = simulate(g, *PHYS); p_ml = p_phys + ml.predict(feats(g, p_phys))
    base = np.full_like(y, g["T"].iloc[0])
    rows.append({"file": g["file"].iloc[0], "Tamb_C": g["Tamb"].iloc[0],
                 "RMSE_physics_C": rmse(p_phys, y), "RMSE_ML_C": rmse(p_ml, y), "RMSE_baseline_C": rmse(base, y),
                 "maxerr_physics_C": float(np.max(np.abs(p_phys-y))), "maxerr_ML_C": float(np.max(np.abs(p_ml-y)))})
    P.append(p_phys); M.append(p_ml); B.append(base); Y.append(y)
results = pd.DataFrame(rows)
overall = {"RMSE_physics_C": rmse(np.concatenate(P), np.concatenate(Y)), "RMSE_ML_C": rmse(np.concatenate(M), np.concatenate(Y)),
           "RMSE_baseline_C": rmse(np.concatenate(B), np.concatenate(Y)), "n_points": int(sum(len(y) for y in Y))}
print(overall); results

In [ ]:
import matplotlib.pyplot as plt
n = min(3, len(test)); fig, axs = plt.subplots(1, n, figsize=(5*n, 3.6), squeeze=False)
for j in range(n):
    g = test[j]; ax = axs[0, j]
    ax.plot(g["t"]/60, g["T"], "k", label="measured"); ax.plot(g["t"]/60, P[j], "--", label="physics (fitted)")
    ax.plot(g["t"]/60, M[j], ":", label="physics + ML correction"); ax.set_xlabel("minutes"); ax.set_ylabel("cell temperature (C)")
    ax.set_title(f'{os.path.basename(g["file"].iloc[0])[:26]}  ({g["Tamb"].iloc[0]:.0f} C)', fontsize=9); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig("held_out_temperature.png", dpi=170); plt.show()

## 8. Save the results
Downloads the files to show judges and to send to me: `metrics.json`, `train_test_split.csv`, `per_file_results.csv`, `held_out_temperature.png`.

In [ ]:
import json, shutil
out = {"dataset": "Panasonic 18650PF (Kollmeyer) DOI 10.17632/wykht8y7tg.1", "split": SPLIT,
       "held_out_temperature_C": (held if SPLIT == "by_temperature" else None),
       "files_train": len(train), "files_test": len(test),
       "fitted_R25_ohm": float(R_fit), "fitted_k_per_C": float(k_fit), "fitted_hA_W_per_K": float(hA_fit), "cell_mass_kg": CELL["mass"], "cell_cp_J_per_kgK": CELL["cp"],
       "overall_test": overall}
json.dump(out, open("metrics.json", "w"), indent=2)
split_df.to_csv("train_test_split.csv", index=False); results.to_csv("per_file_results.csv", index=False)
try:
    from google.colab import files
    for f in ["metrics.json", "train_test_split.csv", "per_file_results.csv", "held_out_temperature.png"]: files.download(f)
except ImportError:
    print("saved in the current folder")